# Masterarbeit – alle Experimente in einem Notebook

**Thema:** Dateneffiziente Personalisierung für webcam-basiertes Eye-Tracking mithilfe vortrainierter Bildverarbeitungsmodelle

Dieses Notebook hat **drei Zellen** zum Ausführen:

| Zelle | Was passiert? | Ergebnis | Dauer (CPU, ca.) |
|---|---|---|---|
| **1** | Daten prüfen, die zwei **Basismodelle** auf GazeCapture trainieren (Split `norm_subject` und `norm_random`) und mit den gespeicherten Modellen vergleichen | `models/base/` (Modelle der Arbeit), `models/base_neu/` (neu trainiert) | 5–20 min |
| **2** | *(optional)* Ergebnisse auf GazeCapture: Tabellen 4.3–4.9 und Abbildung 4.3 | Tabellen, `results/figures/` | Sekunden (Neurechnen: 1–2 h) |
| **3** | **Personalisierung**: die gespeicherten Basismodelle mit den eigenen Sessions 01–09 und 19–27 feinabstimmen | Tabelle 4.13 und Vergleich mit den Werten der Arbeit | 15–40 min beim ersten Mal, danach Minuten |

**So geht's:** Zellen nacheinander mit *Shift + Enter* ausführen. Oben in jeder Zelle stehen die Einstellungen (GROSSBUCHSTABEN).

**Wichtig zur Genauigkeit**
* Basismodelle (Zelle 1) und Personalisierung (Zelle 3) verwenden **feste Seeds**. Auf demselben Rechner kommen dieselben Zahlen heraus. Auf einem anderen Rechner kann es sehr kleine Rundungsunterschiede geben (andere CPU-Bibliotheken).
* Die Hyperparameter-Studie (Zelle 2, `exp.py`) hat **keine** festen Seeds. Beim Neurechnen ändern sich die Einzelwerte daher leicht; Mittelwerte und Aussagen bleiben gleich. Deshalb zeigt Zelle 2 standardmäßig die gespeicherten Werte.
* Die Modelle der Arbeit in `models/base/` werden **nie überschrieben**. Neu trainierte Modelle landen in `models/base_neu/`.

**Fehlermaß:** Diagonalfehler $E_\text{diag} = 100\cdot\text{RMSE}_\text{eukl.}/\sqrt2$ in % (Gleichung 4.1, `metrics.py`). Kleiner ist besser.

## Zelle 1 – Vorbereitung und Basismodelle (GazeCapture)

1. Prüft, ob GazeCapture-Labels und alle 18 Sessions (Bilder + `norm_labels.csv`) da sind.
2. Trainiert die zwei Basismodelle mit der gewählten Einstellung (nur FC, Sigmoid, L1, lr = 1e-4, Batch 32, 1000/200 zufällige Bilder mit Seed 0, Validierung 200 Bilder). Pro Split 5 Seeds; behalten wird der Seed mit dem kleinsten **Validierungs**fehler.
3. Vergleicht die neuen Werte Seed für Seed mit den gespeicherten Modellen (Spalte *Differenz Test* sollte 0 sein).

Gibt es noch gar keine Modelle in `models/base/`, werden sie dort gespeichert.

In [ ]:
# ===================== Zelle 1: Vorbereitung und Basismodelle =====================
NEU_TRAINIEREN = True          # True: Basismodelle neu trainieren (-> models/base_neu) und vergleichen
                               # False: nur die gespeicherten Modelle in models/base anzeigen

import os, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import torch
import train_base
import personal_exp as pe
import auswertung as aw

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
torch.set_num_threads(max(1, os.cpu_count() or 1))
print("Projektordner:", ROOT)
print("Gerät:", "GPU (cuda)" if torch.cuda.is_available() else "CPU", "| Threads:", torch.get_num_threads())

# 1) Daten prüfen
assert Path("dataset/norm_labels.csv").exists(), "GazeCapture fehlt: dataset/norm_labels.csv (siehe README, main.py)"
daten = aw.pruefe_daten()
display(daten)
assert daten.OK.all(), "Für einige Sessions fehlen Bilder oder Labels (siehe Tabelle)."

# 2) Basismodelle
if not (Path("models/base/base_norm_subject.pt").exists() and Path("models/base/base_norm_random.pt").exists()):
    print("Noch keine Basismodelle vorhanden -> Training nach models/base")
    train_base.train_all(out_dir="models/base")
elif NEU_TRAINIEREN:
    print("Neu trainieren nach models/base_neu (die Modelle der Arbeit bleiben unverändert) ...")
    train_base.train_all(out_dir="models/base_neu", skip_existing=False)
    print("\nVergleich alt (models/base) gegen neu (models/base_neu):")
    display(aw.vergleiche_basismodelle("models/base", "models/base_neu").round(5))

print("\nBasismodelle der Arbeit (models/base):")
display(aw.basismodelle_tabelle("models/base"))

## Zelle 2 *(optional)* – Ergebnisse auf GazeCapture (Kapitel 4.2 und 4.3)

* Zeigt die Tabellen der Hyperparameter-Studie (Batchgröße, Lernrate, Schichten, Aktivierung, Verlust, Datenmenge, Split) aus `results/exp_reproduction/exp_results_all.json` – genau die Werte der Arbeit.
* `SWEEPS_NEU = True` rechnet alles neu (1–2 Stunden, mit vollem ResNet). Ergebnis: `exp_results_neu.json`. Ohne feste Seeds weichen die Einzelwerte leicht ab.
* Erzeugt Abbildung 4.3 (räumlicher Fehler) und die Abbildungen der Zielwerte (Abb. 4.1).

In [ ]:
# ===================== Zelle 2: GazeCapture-Ergebnisse (optional) =====================
SWEEPS_NEU = False             # True: Hyperparameter-Studie komplett neu rechnen (1-2 h)
ABBILDUNGEN = True             # True: Abbildungen 4.1 und 4.3 neu zeichnen

import json, subprocess
import exp
import spatial_analysis as sa

datei = "results/exp_reproduction/exp_results_all.json"
if SWEEPS_NEU:
    cfg = {**exp.DEFAULT_CONFIG, "sample_seed": 0, "val_size": 200}
    geraet = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    neu = {"baseline_same_subset": exp.compute_subset_baselines(cfg)}
    for name, lauf in exp.build_experiments(5, cfg, geraet, include_resnet=True).items():
        neu[name] = lauf()
    datei = "results/exp_reproduction/exp_results_neu.json"
    Path(datei).write_text(json.dumps(neu, indent=2, default=float), encoding="utf-8")

print("Quelle:", datei, "| † = gewählte Option, p = Mann-Whitney-U gegen †")
for name, tabelle in aw.hyperparameter_tabellen(datei).items():
    print("\n" + name); display(tabelle)

if ABBILDUNGEN:
    Path("results/figures").mkdir(parents=True, exist_ok=True)
    gt, pr = sa.load_predictions("results/heatmap/base_loss1.csv")       # Standardmodell, L1
    sa.plot_figure4(gt, pr, "results/figures/fig4_spatial_heatmaps.png")
    print("Fehler pro Region (L1):", sa.region_statistics(gt, pr))
    subprocess.run([sys.executable, "plot_targets.py", "--out", "results/figures"], check=True)
    print("Abbildungen gespeichert in results/figures/")

## Zelle 3 – Personalisierung der gespeicherten Modelle

Jede Session wird mit jedem Basismodell einmal personalisiert: Das Basismodell wird geladen, der ResNet-18 bleibt eingefroren, und **nur der Kopf** wird mit den Bildern der Session weiter trainiert (zufälliger Split 70/15/15, Seed 42, Early Stopping auf der Validierung).

* Sessions: **01–09 und 19–27** (18 Sessions), Basismodelle `norm_subject` und `norm_random` → 36 Läufe = Tabelle 4.13 (**E1**).
* Zusätzlich möglich: **E1b** (bereinigte Labels, Tab. 4.14), **E2** (Aufteilung der Daten, Tab. 4.15), **E3** (sequenzieller Split, Tab. 4.17/4.18).
* Beim ersten Mal werden die ResNet-Merkmale aller Bilder einmal berechnet und in `cache/personal/` gespeichert. Danach dauert ein Lauf nur Sekunden.
* Neue Ergebnisse: `results/personal/neu_runs.jsonl`. Die Ergebnisse der Arbeit (`personal_runs.jsonl`) werden **nicht** verändert.
* Am Ende steht jeder neue Lauf neben dem Lauf aus der Arbeit. Die Spalte *Differenz* sollte 0 sein.

In [ ]:
# ===================== Zelle 3: Personalisierung =====================
EXPERIMENTE = ["E1"]           # Tabelle 4.13; zusätzlich möglich: "E1b", "E2", "E3"
BASISMODELLE = "models/base"   # Modelle der Arbeit; oder "models/base_neu" aus Zelle 1
NEU_STARTEN = True             # True: alte neu_runs.jsonl löschen und alles neu rechnen
NUM_WORKERS = 0                # Windows: 0 lassen

import time
ERGEBNIS = Path("results/personal/neu_runs.jsonl")
VORHERSAGEN = Path("results/personal/neu_predictions")
pe.CHECKPOINT = BASISMODELLE + "/base_{base}.pt"
if NEU_STARTEN and ERGEBNIS.exists():
    ERGEBNIS.unlink()
gemeinsam = dict(out=ERGEBNIS, save_predictions=str(VORHERSAGEN), num_workers=NUM_WORKERS)

t0 = time.perf_counter()
if "E1" in EXPERIMENTE or "E3" in EXPERIMENTE or "E1b" in EXPERIMENTE:   # E1b/E3 vergleichen mit E1
    basen = ["norm_subject", "norm_random"] if "E1" in EXPERIMENTE else ["norm_subject"]
    pe.run_series(sessions=["all"], bases=basen, split="random", seeds=[42], **gemeinsam)
if "E1b" in EXPERIMENTE:
    pe.run_series(sessions=["gen4"], labels=["clean"], split="random", seeds=[42], **gemeinsam)
if "E2" in EXPERIMENTE:
    for split in ("random", "sequential"):
        pe.run_series(sessions=["21", "22"], split=split, seeds=[42, 43, 44], gap_sec=1.0, **gemeinsam)
    pe.run_series(split="two_videos", pairs=[("21", "22"), ("22", "21")], seeds=[42, 43, 44], **gemeinsam)
if "E3" in EXPERIMENTE:
    pe.run_series(sessions=["all"], split="sequential", seeds=[42], gap_sec=1.0, **gemeinsam)
print(f"\nfertig nach {(time.perf_counter() - t0) / 60:.1f} min")

# ---------- Tabellen ----------
neu = aw.laeufe_laden(ERGEBNIS)
aw.VORHERSAGEN = VORHERSAGEN
if "E1" in EXPERIMENTE:
    pro_session, pro_gruppe = aw.tabelle_e1(neu)
    print("\nTabelle 4.13 – Personalisierung aller 18 Sessions (Fehler in %)")
    display(pro_session); display(pro_gruppe)
    print(aw.tests_e1(neu))
    aw.abbildung_sessions(neu, "results/figures/fig_sessions_baseline_neu.png")
if "E1b" in EXPERIMENTE:
    print("\nTabelle 4.14 – Gen 4: Original gegen bereinigte Labels"); display(aw.tabelle_e1b(neu))
if "E2" in EXPERIMENTE:
    print("\nTabelle 4.15 – Aufteilung der Daten (Sessions 21/22, 3 Seeds)"); display(aw.tabelle_e2(neu))
if "E3" in EXPERIMENTE:
    design, regionen = aw.tabelle_e3(neu)
    print("\nTabelle 4.17 – Gestaltung der Sitzung"); display(design)
    print("Tabelle 4.18 – Fehler pro Bildschirmregion (in Klammern: Anzahl Testbilder)"); display(regionen)

# ---------- Vergleich mit den Ergebnissen der Arbeit ----------
vergleich, groesste = aw.vergleiche_mit_referenz(ERGEBNIS)
print(f"\nVergleich mit results/personal/personal_runs.jsonl: {len(vergleich)} Läufe, "
      f"größte Abweichung im Testfehler = {groesste:.4f} Prozentpunkte")
display(vergleich)